In [6]:
# ==============================================================================
# 1. INSTALAÇÃO DE DEPENDÊNCIAS (Sem conectores Java do S3)
# ==============================================================================
!pip install pyspark boto3 pandas -q


import os
import boto3
import pandas as pd
from datetime import datetime, timezone
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# ==============================================================================
# 2. INICIALIZAÇÃO DO SPARK (Simples e Leve)
# ==============================================================================
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Ingestao-Colab-Bronze") \
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic") \
    .getOrCreate()

# ==============================================================================
# 3. CONFIGURAÇÃO DO CLIENTE AWS NATIVO (Boto3)
# ==============================================================================
AWS_ACCESS_KEY = "AWS_KEY"
AWS_SECRET_KEY = "AWS_SECRET"
AWS_SESSION_TOKEN = "SESSION_TOKEN"
# Cria o cliente S3 nativo do Python (não passa pelo Java/Hadoop)
if AWS_SESSION_TOKEN and AWS_SESSION_TOKEN != "SESSION_TOKEN":
    s3_client = boto3.client(
        's3',
        aws_access_key_id=AWS_ACCESS_KEY,
        aws_secret_access_key=AWS_SECRET_KEY,
        aws_session_token=AWS_SESSION_TOKEN
    )
else:
    s3_client = boto3.client(
        's3',
        aws_access_key_id=AWS_ACCESS_KEY,
        aws_secret_access_key=AWS_SECRET_KEY
    )

# ==============================================================================
# 4. VARIÁVEIS GLOBAIS DE TEMPO E BUCKETS
# ==============================================================================
INGESTION_TS = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
INGESTION_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")
ANOMESDIA = datetime.now(timezone.utc).strftime("%Y%m%d")

BUCKET_PRINICIPAL = "fiap-datalake-fase2"
PASTA_RAW = "raw"
PASTA_BRONZE = "bronze"
PASTA_SILVER = "silver"
PASTA_GOLD = "gold"
QUALITY_PATH = "quality_reports"
print("✅ Ambiente Spark + Boto3 configurado com sucesso!")

✅ Ambiente Spark + Boto3 configurado com sucesso!


In [7]:
# ==============================================================================
# CÉLULA 2: MAPEAMENTO DE SCHEMAS E ARQUIVOS REAIS DO S3
# ==============================================================================
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# Seus nomes de entidades organizadas (como vão ficar as pastas na Bronze)
ENTIDADES = [
    "meta_brasil", "meta_uf", "meta_municipio",
    "municipio", "uf", "alunos_2023", "alunos_2024", "alunos_2025"
]

# DICIONÁRIO DE DEPARA: Relaciona a entidade com o nome REAL do arquivo no S3
ARQUIVOS_MAP = {
    "meta_brasil": "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv",
    "meta_uf": "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv",
    "meta_municipio": "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv",
    "municipio": "br_inep_avaliacao_alfabetizacao_municipio.csv",
    "uf": "br_inep_avaliacao_alfabetizacao_uf.csv",
    "alunos_2023": "TS_ALUNO_2023.csv",
    "alunos_2024": "TS_ALUNO_2024.csv",
    "alunos_2025": "TS_ALUNO_2025.csv"
}

# Dicionário de schemas
SCHEMAS_MAP = {
    "meta_brasil": StructType([
        StructField("ano", IntegerType(), True),
        StructField("rede", StringType(), True),
        StructField("taxa_alfabetizacao", DoubleType(), True),
        StructField("meta_alfabetizacao_2024", DoubleType(), True),
        StructField("meta_alfabetizacao_2025", DoubleType(), True),
        StructField("meta_alfabetizacao_2026", DoubleType(), True),
        StructField("meta_alfabetizacao_2027", DoubleType(), True),
        StructField("meta_alfabetizacao_2028", DoubleType(), True),
        StructField("meta_alfabetizacao_2029", DoubleType(), True),
        StructField("meta_alfabetizacao_2030", DoubleType(), True),
        StructField("percentual_participacao", DoubleType(), True)
    ]),
    "meta_uf": StructType([
        StructField("ano", IntegerType(), True),
        StructField("sigla_uf", StringType(), True),
        StructField("rede", StringType(), True),
        StructField("taxa_alfabetizacao", DoubleType(), True),
        StructField("meta_alfabetizacao_2024", DoubleType(), True),
        StructField("meta_alfabetizacao_2025", DoubleType(), True),
        StructField("meta_alfabetizacao_2026", DoubleType(), True),
        StructField("meta_alfabetizacao_2027", DoubleType(), True),
        StructField("meta_alfabetizacao_2028", DoubleType(), True),
        StructField("meta_alfabetizacao_2029", DoubleType(), True),
        StructField("meta_alfabetizacao_2030", DoubleType(), True),
        StructField("percentual_participacao", DoubleType(), True)
    ]),
    "meta_municipio": StructType([
        StructField("ano", IntegerType(), True),
        StructField("id_municipio", IntegerType(), True),
        StructField("rede", StringType(), True),
        StructField("taxa_alfabetizacao", DoubleType(), True),
        StructField("meta_alfabetizacao_2024", DoubleType(), True),
        StructField("meta_alfabetizacao_2025", DoubleType(), True),
        StructField("meta_alfabetizacao_2026", DoubleType(), True),
        StructField("meta_alfabetizacao_2027", DoubleType(), True),
        StructField("meta_alfabetizacao_2028", DoubleType(), True),
        StructField("meta_alfabetizacao_2029", DoubleType(), True),
        StructField("meta_alfabetizacao_2030", DoubleType(), True),
        StructField("nivel_alfabetizacao", DoubleType(), True),
        StructField("percentual_participacao", DoubleType(), True)
    ]),
    "municipio": StructType([
        StructField("ano", IntegerType(), True),
        StructField("id_municipio", IntegerType(), True),
        StructField("serie", IntegerType(), True),
        StructField("rede", IntegerType(), True),
        StructField("taxa_alfabetizacao", DoubleType(), True),
        StructField("media_portugues", DoubleType(), True),
        StructField("proporcao_aluno_nivel_0", DoubleType(), True),
        StructField("proporcao_aluno_nivel_1", DoubleType(), True),
        StructField("proporcao_aluno_nivel_2", DoubleType(), True),
        StructField("proporcao_aluno_nivel_3", DoubleType(), True),
        StructField("proporcao_aluno_nivel_4", DoubleType(), True),
        StructField("proporcao_aluno_nivel_5", DoubleType(), True),
        StructField("proporcao_aluno_nivel_6", DoubleType(), True),
        StructField("proporcao_aluno_nivel_7", DoubleType(), True),
        StructField("proporcao_aluno_nivel_8", DoubleType(), True)
    ]),
    "uf": StructType([
          StructField("ano", IntegerType(), True),
          StructField("sigla_uf", StringType(), True),
          StructField("serie", IntegerType(), True),
          StructField("rede", IntegerType(), True),
          StructField("taxa_alfabetizacao", DoubleType(), True),
          StructField("media_portugues", DoubleType(), True),
          StructField("proporcao_aluno_nivel_0", DoubleType(), True),
          StructField("proporcao_aluno_nivel_1", DoubleType(), True),
          StructField("proporcao_aluno_nivel_2", DoubleType(), True),
          StructField("proporcao_aluno_nivel_3", DoubleType(), True),
          StructField("proporcao_aluno_nivel_4", DoubleType(), True),
          StructField("proporcao_aluno_nivel_5", DoubleType(), True),
          StructField("proporcao_aluno_nivel_6", DoubleType(), True),
          StructField("proporcao_aluno_nivel_7", DoubleType(), True),
          StructField("proporcao_aluno_nivel_8", DoubleType(), True)
    ]),
    "alunos_2023": StructType([
        StructField("NU_ANO_AVALIACAO", IntegerType(), True),
        StructField("CO_UF", IntegerType(), True),
        StructField("SG_UF", StringType(), True),
        StructField("ID_ALUNO", IntegerType(), True),
        StructField("TP_SERIE", IntegerType(), True),
        StructField("ID_ESCOLA", IntegerType(), True),
        StructField("TP_DEPENDENCIA", IntegerType(), True),
        StructField("CO_MUNICIPIO", IntegerType(), True),
        StructField("NO_MUNICIPIO", StringType(), True),
        StructField("IN_PRESENCA_LP", IntegerType(), True),
        StructField("IN_PREENCHIMENTO_LP", IntegerType(), True),
        StructField("CO_CADERNO_LP", IntegerType(), True),
        StructField("VL_PESO_ALUNO_LP", DoubleType(), True),
        StructField("VL_PROFICIENCIA_LP", DoubleType(), True),
        StructField("IN_ALFABETIZADO", IntegerType(), True)
    ]),
    "alunos_2024": StructType([
        StructField("NU_ANO_AVALIACAO", IntegerType(), True),
        StructField("CO_UF", IntegerType(), True),
        StructField("SG_UF", StringType(), True),
        StructField("ID_ALUNO", IntegerType(), True),
        StructField("TP_SERIE", IntegerType(), True),
        StructField("ID_ESCOLA", IntegerType(), True),
        StructField("TP_DEPENDENCIA", IntegerType(), True),
        StructField("CO_MUNICIPIO", IntegerType(), True),
        StructField("NO_MUNICIPIO", StringType(), True),
        StructField("IN_PRESENCA_LP", IntegerType(), True),
        StructField("IN_PREENCHIMENTO_LP", IntegerType(), True),
        StructField("CO_CADERNO_LP", IntegerType(), True),
        StructField("VL_PESO_ALUNO_LP", DoubleType(), True),
        StructField("VL_PROFICIENCIA_LP", DoubleType(), True),
        StructField("IN_ALFABETIZADO", IntegerType(), True)
    ]),
    "alunos_2025": StructType([
        StructField("NU_ANO_AVALIACAO", IntegerType(), True),
        StructField("CO_UF", IntegerType(), True),
        StructField("SG_UF", StringType(), True),
        StructField("ID_ALUNO", IntegerType(), True),
        StructField("TP_SERIE", IntegerType(), True),
        StructField("ID_ESCOLA", IntegerType(), True),
        StructField("TP_DEPENDENCIA", IntegerType(), True),
        StructField("CO_MUNICIPIO", IntegerType(), True),
        StructField("NO_MUNICIPIO", StringType(), True),
        StructField("IN_PRESENCA_LP", IntegerType(), True),
        StructField("IN_PREENCHIMENTO_LP", IntegerType(), True),
        StructField("CO_CADERNO_LP", IntegerType(), True),
        StructField("VL_PESO_ALUNO_LP", DoubleType(), True),
        StructField("VL_PROFICIENCIA_LP", DoubleType(), True),
        StructField("IN_ALFABETIZADO", IntegerType(), True)
    ]),

}
print("✅ Mapeamento de arquivos físicos carregado com sucesso!")

✅ Mapeamento de arquivos físicos carregado com sucesso!


In [8]:
def fazer_upload_pasta_s3(pasta_local, bucket, prefixo_s3):
    """
    Função auxiliar para varrer a pasta local do Colab e enviar os arquivos ao S3
    """
    import os
    for raiz, diretorios, arquivos in os.walk(pasta_local):
        for arquivo in arquivos:
            caminho_completo_local = os.path.join(raiz, arquivo)
            # Calcula o caminho relativo para manter a estrutura de pastas no S3
            caminho_relativo = os.path.relpath(caminho_completo_local, pasta_local)
            key_s3 = os.path.join(prefixo_s3, caminho_relativo).replace("\\", "/")

            # Envia o arquivo para o S3 via Boto3
            s3_client.upload_file(caminho_completo_local, bucket, key_s3)

def executar_pipeline_bronze(entidade, bucket, pasta_in, pasta_out):
    """
    Lê o CSV do S3 tratando encoding e identificando o separador (, ou ;) de forma
    automática, aplica tipagem via .cast() tratando strings 'NaN'/nulas, adiciona metadados,
    grava em Parquet localmente e faz o upload dos arquivos gerados de volta para o S3.
    """
    print(f"\n--- Iniciando processamento da entidade: {entidade} ---")

    # Busca o nome real do arquivo no dicionário mapeado
    nome_arquivo_real = ARQUIVOS_MAP.get(entidade, f"{entidade}.csv")

    key_input_csv = f"{pasta_in}/{nome_arquivo_real}"

    # Define a estrutura de partições
    estrutura_destino = f"{entidade}/anomesdia={ANOMESDIA}"
    path_local_output = f"./output/{pasta_out}/{estrutura_destino}"
    key_s3_output = f"{pasta_out}/{estrutura_destino}"

    print(f"[RAW] Baixando objeto do S3 via Boto3: {key_input_csv}")

    # 1. Leitura do S3 tratando dinamicamente Encoding e Separador
    try:
        obj = s3_client.get_object(Bucket=bucket, Key=key_input_csv)
        corpo_arquivo = obj['Body'].read()

        try:
            # Tenta UTF-8 descobrindo o separador automaticamente
            pdf = pd.read_csv(pd.io.common.BytesIO(corpo_arquivo), sep=None, engine='python', encoding="utf-8", dtype=str)
        except UnicodeDecodeError:
            print(f"[AVISO] Falha ao decodificar em UTF-8 para '{entidade}'. Tentando padrão ISO-8859-1 (Latin1)...")
            # Fallback para Latin1 descobrindo o separador automaticamente
            pdf = pd.read_csv(pd.io.common.BytesIO(corpo_arquivo), sep=None, engine='python', encoding="ISO-8859-1", dtype=str)

    except Exception as e:
        print(f"[ERRO] Falha ao ler ou decodificar arquivo do S3 via Boto3: {str(e)}")
        raise e

    # 2. Captura o schema específico se existir, caso contrário lê genérico
    schema_especifico = SCHEMAS_MAP.get(entidade, None)

    # Convertendo o DataFrame do Pandas para Spark DataFrame
    if schema_especifico:
        print(f"[RAW] Schema explícito encontrado para '{entidade}'. Ajustando tipos e tratando NaNs...")

        # Cria o DataFrame temporário flexível
        df_temp = spark.createDataFrame(pdf)

        # Intercepta strings textuais nulas/NaN antes de realizar o .cast()
        expressoes_tipo = [
            F.when(F.col(campo.name).isin(["NaN", "nan", "None", "NULL", "nan"]), F.lit(None))
             .otherwise(F.col(campo.name))
             .cast(campo.dataType)
             .alias(campo.name)
            for campo in schema_especifico
        ]
        df_raw = df_temp.select(*expressoes_tipo)
    else:
        print(f"[RAW] Nenhum schema mapeado para '{entidade}'. Convertendo como texto.")
        df_raw = spark.createDataFrame(pdf)

    # 3. Adição de Metadados de Auditoria
    print("[BRONZE] Aplicando metadados de auditoria...")
    path_fake_s3 = f"s3://{bucket}/{key_input_csv}"
    df_bronze = (df_raw
        .withColumn("_ingestion_timestamp", F.lit(INGESTION_TS))
        .withColumn("_ingestion_date",      F.lit(INGESTION_DATE))
        .withColumn("_source_path",         F.lit(path_fake_s3))
        .withColumn("_source_entity",       F.lit(entidade))
        .withColumn("_environment",         F.lit("dev"))
    )

    # 4. Escrita no formato Parquet (Local no ambiente do Colab)
    print(f"[BRONZE] Salvando Parquet temporariamente no Colab...")
    df_bronze.write.mode("overwrite").parquet(path_local_output)

    # 5. Upload da estrutura Parquet gerada de volta para o S3
    print(f"[S3-UPLOAD] Enviando arquivos Parquet para o S3 em: s3://{bucket}/{key_s3_output}/")
    try:
        fazer_upload_pasta_s3(path_local_output, bucket, key_s3_output)
        print(f"[S3-UPLOAD] Upload concluído com sucesso!")
    except Exception as e:
        print(f"[ERRO] Falha ao enviar arquivos para o S3: {str(e)}")
        raise e

    print(f"[SUCESSO] Entidade '{entidade}' processada e salva no S3. Registros: {df_bronze.count()}")

In [9]:
# Execução em loop para as tabelas
for entidade in ENTIDADES:
    try:
        executar_pipeline_bronze(
            entidade=entidade,
            bucket=BUCKET_PRINICIPAL,
            pasta_in=PASTA_RAW,
            pasta_out=PASTA_BRONZE
        )
    except Exception as e:
        print(f"[ERRO] Falha ao processar a entidade {entidade}: {str(e)}")


--- Iniciando processamento da entidade: meta_brasil ---
[RAW] Baixando objeto do S3 via Boto3: raw/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv
[RAW] Schema explícito encontrado para 'meta_brasil'. Ajustando tipos e tratando NaNs...
[BRONZE] Aplicando metadados de auditoria...
[BRONZE] Salvando Parquet temporariamente no Colab...
[S3-UPLOAD] Enviando arquivos Parquet para o S3 em: s3://fiap-datalake-fase2/bronze/meta_brasil/anomesdia=20260627/
[S3-UPLOAD] Upload concluído com sucesso!
[SUCESSO] Entidade 'meta_brasil' processada e salva no S3. Registros: 3

--- Iniciando processamento da entidade: meta_uf ---
[RAW] Baixando objeto do S3 via Boto3: raw/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv
[RAW] Schema explícito encontrado para 'meta_uf'. Ajustando tipos e tratando NaNs...
[BRONZE] Aplicando metadados de auditoria...
[BRONZE] Salvando Parquet temporariamente no Colab...
[S3-UPLOAD] Enviando arquivos Parquet para o S3 em: s3://fiap-datalake-fase2/bro

In [10]:
# ==============================================================================
# CAMADA SILVER - CÉLULA 1: DOWNLOAD MULTI-ENTIDADE DA BRONZE VIA BOTO3
# ==============================================================================
import os

# Configurações de diretórios globais da Silver
PASTA_SILVER = "silver"

print(f"--- [SILVER] Iniciando download em lote para as {len(ENTIDADES)} entidades ---")

# Loop para baixar os dados de todas as tabelas listadas na sua Bronze
for entidade in ENTIDADES:
    prefixo_s3_bronze = f"{PASTA_BRONZE}/{entidade}/anomesdia={ANOMESDIA}/"
    path_bronze_local = f"./output/{PASTA_BRONZE}/{entidade}/anomesdia={ANOMESDIA}/"

    print(f"\n📡 Buscando arquivos para '{entidade}' em: s3://{BUCKET_PRINICIPAL}/{prefixo_s3_bronze}")
    os.makedirs(path_bronze_local, exist_ok=True)

    try:
        response = s3_client.list_objects_v2(Bucket=BUCKET_PRINICIPAL, Prefix=prefixo_s3_bronze)

        if 'Contents' in response:
            for obj in response['Contents']:
                file_key = obj['Key']
                if file_key.endswith('/'):
                    continue

                nome_arquivo = os.path.basename(file_key)
                destino_local = os.path.join(path_bronze_local, nome_arquivo)

                print(f"  📥 Baixando Parquet: {nome_arquivo}...")
                s3_client.download_file(BUCKET_PRINICIPAL, file_key, destino_local)
        else:
            print(f"  ⚠️ Atenção: Nenhum arquivo encontrado para a entidade '{entidade}'.")
    except Exception as e:
        print(f"  ❌ Erro ao baixar dados de '{entidade}': {str(e)}")

print("\n✅ Processo de download em lote finalizado!")

--- [SILVER] Iniciando download em lote para as 8 entidades ---

📡 Buscando arquivos para 'meta_brasil' em: s3://fiap-datalake-fase2/bronze/meta_brasil/anomesdia=20260627/
  📥 Baixando Parquet: ._SUCCESS.crc...
  📥 Baixando Parquet: .part-00000-1db07064-762b-46a0-9e84-27716395e6f8-c000.snappy.parquet.crc...
  📥 Baixando Parquet: .part-00000-652f16eb-ed5b-488c-a88c-cd7a0e7debc1-c000.snappy.parquet.crc...
  📥 Baixando Parquet: .part-00001-1db07064-762b-46a0-9e84-27716395e6f8-c000.snappy.parquet.crc...
  📥 Baixando Parquet: .part-00001-652f16eb-ed5b-488c-a88c-cd7a0e7debc1-c000.snappy.parquet.crc...
  📥 Baixando Parquet: _SUCCESS...
  📥 Baixando Parquet: part-00000-1db07064-762b-46a0-9e84-27716395e6f8-c000.snappy.parquet...
  📥 Baixando Parquet: part-00000-652f16eb-ed5b-488c-a88c-cd7a0e7debc1-c000.snappy.parquet...
  📥 Baixando Parquet: part-00001-1db07064-762b-46a0-9e84-27716395e6f8-c000.snappy.parquet...
  📥 Baixando Parquet: part-00001-652f16eb-ed5b-488c-a88c-cd7a0e7debc1-c000.snappy.pa

In [14]:
# ==============================================================================
# CAMADA SILVER - TRATAMENTO DE VALORES NULOS E DUPLICIDADES EM LOTE
# ==============================================================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType, FloatType
import shutil
import os

# ------------------------------------------------------------------------------
# CONFIGURAÇÕES E MAPEAMENTOS
# ------------------------------------------------------------------------------
CHAVES_NEGOCIO_MAP = {
    "alunos_2023": ["nu_ano_avaliacao", "id_aluno"],
    "alunos_2024": ["nu_ano_avaliacao", "id_aluno"],
    "alunos_2025": ["nu_ano_avaliacao", "id_aluno"],
    "meta_brasil": ["ano", "rede"],
    "meta_uf": ["ano", "sg_uf"],
    "meta_municipio": ["ano", "id_municipio"],
    "municipio": ["ano", "id_municipio"],
    "uf": ["ano", "sg_uf"]

}

# ------------------------------------------------------------------------------
# FUNÇÕES UTILIÁRIAS
# ------------------------------------------------------------------------------
def padronizar_schema_e_nomes(df, entidade):
    print(f"  📝 Padronizando nomes de colunas e schema para a entidade '{entidade}'...")

    colunas_transformadas = []
    for col in df.columns:
        # 1. Mudamos de .lower() para .upper()
        c_upper = col.upper()

        # 2. Ajustamos a regra de mapeamento para bater com o padrão maiúsculo
        if c_upper == "SIGLA_UF":
            c_upper = "SG_UF"

        colunas_transformadas.append(F.col(col).alias(c_upper))

    df = df.select(*colunas_transformadas)

    expressoes_tipo = []
    colunas_string_convertidas = 0

    for campo in df.schema:
        if isinstance(campo.dataType, StringType):
            colunas_string_convertidas += 1
            expressoes_tipo.append(F.col(campo.name).cast(StringType()).alias(campo.name))
        else:
            expressoes_tipo.append(F.col(campo.name))

    if colunas_string_convertidas > 0:
        print(f"  🔤 Garantindo tipo String para {colunas_string_convertidas} coluna(s)...")

    return df.select(*expressoes_tipo)
def aplicar_desduplicacao(df, chaves_obrigatorias):
    chaves_ajustadas = [
        "sg_uf" if (c == "sigla_uf" and "sg_uf" in df.columns) else c
        for c in chaves_obrigatorias
    ]

    print(f"  🧹 Removendo nulos nas chaves: {chaves_ajustadas}")
    df_limpo = df.dropna(subset=chaves_ajustadas)

    print("  ✨ Aplicando desduplicação por Window Function (Mais recente)...")
    window_spec = Window.partitionBy(*chaves_ajustadas).orderBy(F.col("_ingestion_timestamp").desc())

    return (df_limpo
            .withColumn("row_num", F.row_number().over(window_spec))
            .filter(F.col("row_num") == 1)
            .drop("row_num"))

def aplicar_regras_especificas_alunos_2025(df):
    colunas_para_dropar = [
        "co_bloco_1", "tx_resposta_bloco_1", "tx_gabarito_bloco_1",
        "co_bloco_2", "tx_resposta_bloco_2", "tx_gabarito_bloco_2",
        "co_bloco_3", "tx_resposta_bloco_3", "tx_gabarito_bloco_3",
        "co_bloco_4", "tx_resposta_bloco_4", "tx_gabarito_bloco_4"
    ]
    colunas_existentes = [c for c in colunas_para_dropar if c in df.columns]

    if colunas_existentes:
        print(f"  🔥 [CUSTOM] Removendo {len(colunas_existentes)} colunas de blocos em alunos_2025")
        df = df.drop(*colunas_existentes)

    print("  🔢 [CUSTOM] Convertendo colunas específicas para Inteiro...")
    colunas_para_inteiro = ["id_escola", "tp_dependencia", "co_municipio"]
    for col_num in colunas_para_inteiro:
        if col_num in df.columns:
            df = df.withColumn(col_num, F.col(col_num).cast("int"))

    return df

def tratar_nulos_e_numericos(df):
    # ✍️ 1. Substituindo valores nulos/vazios em colunas de texto...
    colunas_string = [c.name for c in df.schema.fields if isinstance(c.dataType, StringType)]
    for col_name in colunas_string:
        df = df.withColumn(
            col_name,
            F.when(F.trim(F.col(col_name)) == "", F.lit(None)).otherwise(F.col(col_name))
        )
    df = df.fillna("N/A", subset=colunas_string)

    # 📐 2. Mapeando colunas decimais E inteiras
    tipos_numericos = ["double", "float", "int"]

    colunas_numericas = [
        c.name for c in df.schema.fields
        if any(tipo in c.dataType.simpleString() for tipo in tipos_numericos)
    ]

    if colunas_numericas:
        print(f" 📐 Formatando nulos e arredondando {len(colunas_numericas)} coluna(s) numérica(s)...")
        for col_num in colunas_numericas:
            df = df.withColumn(
                col_num,
                F.when(
                    F.col(col_num).isNull() | (F.trim(F.col(col_num).cast("string")) == ""),
                    F.lit(None)
                ).otherwise(F.round(F.col(col_num), 2))
            )

    return df

# ------------------------------------------------------------------------------
# LOOP PRINCIPAL DE EXECUÇÃO
# ------------------------------------------------------------------------------
print("--- [SILVER] Iniciando transformações e qualidade de dados ---")

for entidade in ENTIDADES:
    print(f"\n🛠️ Processando qualidade da entidade: {entidade}")

    path_bronze_local = f"./output/{PASTA_BRONZE}/{entidade}/anomesdia={ANOMESDIA}/"
    path_silver_local = f"./output/{PASTA_SILVER}/{entidade}/anomesdia={ANOMESDIA}/"

    if not os.path.exists(path_bronze_local) or not os.listdir(path_bronze_local):
        print(f"  ⏩ Pulando '{entidade}' pois não há arquivos locais baixados.")
        continue

    # 1. Leitura
    df_silver = spark.read.parquet(path_bronze_local)
    print(f"  📊 Registros originais na Bronze: {df_silver.count()}")

    # 2. Padronização
    df_silver = padronizar_schema_e_nomes(df_silver, entidade)

    # 3. Deduplicação
    chaves_obrigatorias = CHAVES_NEGOCIO_MAP.get(entidade)
    df_silver = aplicar_desduplicacao(df_silver, chaves_obrigatorias)

    # 4. Regras Customizadas
    if entidade == "alunos_2025":
        df_silver = aplicar_regras_especificas_alunos_2025(df_silver)

    colunas_alvo_string = ["sg_uf", "no_municipio", "rede"]
    colunas_a_converter = [c for c in colunas_alvo_string if c in df_silver.columns]

    if colunas_a_converter:
        print(f"  🔤 [CUSTOM] Convertendo colunas para string: {colunas_a_converter}")
        for col_str in colunas_a_converter:
            df_silver = df_silver.withColumn(col_str, F.col(col_str).cast("string"))


    # 5. Higienização
    df_silver = tratar_nulos_e_numericos(df_silver)

    # 6. Metadados
    df_silver = (df_silver
        .withColumn("_silver_processed_at", F.current_timestamp())
        .withColumn("_pipeline_version", F.lit("v1.2_silver_uppercase"))
    )

    # --------------------------------------------------------------------------
    # 7. CORREÇÃO: PADRONIZAÇÃO EM MAIÚSCULO E GRAVAÇÃO
    # --------------------------------------------------------------------------
    # Aplicamos a conversão diretamente no df_silver antes de salvar
    df_silver = df_silver.toDF(*[c.upper() for c in df_silver.columns])

    if os.path.exists(path_silver_local):
        shutil.rmtree(path_silver_local)

    print(f"  💾 Gravando Parquet em: {path_silver_local}")
    df_silver.write.mode("overwrite").parquet(path_silver_local)
    print(f"  ✅ Concluído! Registros finais na Silver: {df_silver.count()}")

print("\n🎉 Todas as tabelas foram tratadas, padronizadas em MAIÚSCULO e armazenadas localmente!")

--- [SILVER] Iniciando transformações e qualidade de dados ---

🛠️ Processando qualidade da entidade: meta_brasil
  📊 Registros originais na Bronze: 6
  📝 Padronizando nomes de colunas e schema para a entidade 'meta_brasil'...
  🔤 Garantindo tipo String para 6 coluna(s)...
  🧹 Removendo nulos nas chaves: ['ano', 'rede']
  ✨ Aplicando desduplicação por Window Function (Mais recente)...
 📐 Formatando nulos e arredondando 10 coluna(s) numérica(s)...
  💾 Gravando Parquet em: ./output/silver/meta_brasil/anomesdia=20260627/
  ✅ Concluído! Registros finais na Silver: 3

🛠️ Processando qualidade da entidade: meta_uf
  📊 Registros originais na Bronze: 108
  📝 Padronizando nomes de colunas e schema para a entidade 'meta_uf'...
  🔤 Garantindo tipo String para 7 coluna(s)...
  🧹 Removendo nulos nas chaves: ['ano', 'sg_uf']
  ✨ Aplicando desduplicação por Window Function (Mais recente)...
 📐 Formatando nulos e arredondando 10 coluna(s) numérica(s)...
  💾 Gravando Parquet em: ./output/silver/meta_uf

In [15]:
# ==============================================================================
# CAMADA SILVER - CÉLULA 3: UPLOAD EM LOTE DOS DADOS TRATADOS PARA O S3 (SILVER)
# ==============================================================================

print("--- [SILVER] Iniciando upload em lote para a camada Silver no S3 ---")

for entidade in ENTIDADES:
    path_silver_local = f"./output/{PASTA_SILVER}/{entidade}/anomesdia={ANOMESDIA}/"
    key_s3_silver_output = f"{PASTA_SILVER}/{entidade}/anomesdia={ANOMESDIA}/"

    # Garante que só tentará subir se a pasta local foi criada no passo anterior
    if not os.path.exists(path_silver_local):
        continue

    print(f"\n🚀 Enviando '{entidade}' para: s3://{BUCKET_PRINICIPAL}/{key_s3_silver_output}/")

    try:
        fazer_upload_pasta_s3(
            pasta_local=path_silver_local,
            bucket=BUCKET_PRINICIPAL,
            prefixo_s3=key_s3_silver_output
        )
        print(f"  🎉 Upload de '{entidade}' concluído com sucesso!")
    except Exception as e:
        print(f"  ❌ Falha ao enviar arquivos de '{entidade}' para a Silver: {str(e)}")

print("\n🏆 Pipeline da Camada Silver finalizado com sucesso para todas as entidades!")

--- [SILVER] Iniciando upload em lote para a camada Silver no S3 ---

🚀 Enviando 'meta_brasil' para: s3://fiap-datalake-fase2/silver/meta_brasil/anomesdia=20260627//
  🎉 Upload de 'meta_brasil' concluído com sucesso!

🚀 Enviando 'meta_uf' para: s3://fiap-datalake-fase2/silver/meta_uf/anomesdia=20260627//
  🎉 Upload de 'meta_uf' concluído com sucesso!

🚀 Enviando 'meta_municipio' para: s3://fiap-datalake-fase2/silver/meta_municipio/anomesdia=20260627//
  🎉 Upload de 'meta_municipio' concluído com sucesso!

🚀 Enviando 'municipio' para: s3://fiap-datalake-fase2/silver/municipio/anomesdia=20260627//
  🎉 Upload de 'municipio' concluído com sucesso!

🚀 Enviando 'uf' para: s3://fiap-datalake-fase2/silver/uf/anomesdia=20260627//
  🎉 Upload de 'uf' concluído com sucesso!

🚀 Enviando 'alunos_2023' para: s3://fiap-datalake-fase2/silver/alunos_2023/anomesdia=20260627//
  🎉 Upload de 'alunos_2023' concluído com sucesso!

🚀 Enviando 'alunos_2024' para: s3://fiap-datalake-fase2/silver/alunos_2024/ano

In [16]:
print("--- 🔍 VISUALIZANDO OS 3 PRIMEIROS REGISTROS DA CAMADA SILVER ---")

for entidade in CHAVES_NEGOCIO_MAP.keys():
    path_silver_local = f"./output/{PASTA_SILVER}/{entidade}/anomesdia={ANOMESDIA}/"

    if os.path.exists(path_silver_local) and os.listdir(path_silver_local):
        print(f"\n📊 Tabela: {entidade}")

        # 1. Lê o Parquet da Silver mantendo o objeto DataFrame do PySpark
        df_spark = spark.read.parquet(path_silver_local)

        # 2. Mostra a tipagem real das colunas no formato PySpark (evita conversão para tipos do Pandas)
        print("📋 Tipagem real das colunas (PySpark):")
        df_spark.printSchema()

        # 3. Exibe os 3 primeiros registros nativos (valores nulos aparecerão como 'null')
        print("👀 Primeiros 3 registros:")
        df_spark.show(3, truncate=False)

    else:
        print(f"\n⚠️ Tabela '{entidade}' não encontrada ou vazia no caminho local.")

--- 🔍 VISUALIZANDO OS 3 PRIMEIROS REGISTROS DA CAMADA SILVER ---

📊 Tabela: meta_brasil
📋 Tipagem real das colunas (PySpark):
root
 |-- ANO: integer (nullable = true)
 |-- REDE: string (nullable = true)
 |-- TAXA_ALFABETIZACAO: double (nullable = true)
 |-- META_ALFABETIZACAO_2024: double (nullable = true)
 |-- META_ALFABETIZACAO_2025: double (nullable = true)
 |-- META_ALFABETIZACAO_2026: double (nullable = true)
 |-- META_ALFABETIZACAO_2027: double (nullable = true)
 |-- META_ALFABETIZACAO_2028: double (nullable = true)
 |-- META_ALFABETIZACAO_2029: double (nullable = true)
 |-- META_ALFABETIZACAO_2030: double (nullable = true)
 |-- PERCENTUAL_PARTICIPACAO: double (nullable = true)
 |-- _INGESTION_TIMESTAMP: string (nullable = true)
 |-- _INGESTION_DATE: string (nullable = true)
 |-- _SOURCE_PATH: string (nullable = true)
 |-- _SOURCE_ENTITY: string (nullable = true)
 |-- _ENVIRONMENT: string (nullable = true)
 |-- _SILVER_PROCESSED_AT: timestamp (nullable = true)
 |-- _PIPELINE_VERS

In [17]:
# ==============================================================================
# CAMADA GOLD - CÉLULA 1: DOWNLOAD MULTI-ENTIDADE DA SILVER VIA BOTO3
# ==============================================================================
import os

# Definindo as entidades da Silver que precisamos para construir nossas visões de negócio
ENTIDADES_SILVER_NECESSARIAS = ["alunos_2023", "alunos_2024", "alunos_2025", "meta_municipio"]

print(f"--- [GOLD] Iniciando download em lote da camada Silver para as tabelas analíticas ---")

for entidade in ENTIDADES_SILVER_NECESSARIAS:
    prefixo_s3_silver = f"{PASTA_SILVER}/{entidade}/anomesdia={ANOMESDIA}/"
    path_silver_local = f"./output/{PASTA_SILVER}/{entidade}/anomesdia={ANOMESDIA}/"

    print(f"\n📡 Buscando arquivos para '{entidade}' em: s3://{BUCKET_PRINICIPAL}/{prefixo_s3_silver}")
    os.makedirs(path_silver_local, exist_ok=True)

    try:
        response = s3_client.list_objects_v2(Bucket=BUCKET_PRINICIPAL, Prefix=prefixo_s3_silver)

        if 'Contents' in response:
            for obj in response['Contents']:
                file_key = obj['Key']
                if file_key.endswith('/') or os.path.basename(file_key).startswith('.'):
                    continue  # Ignora pastas vazias e arquivos ocultos (.part...crc)

                nome_arquivo = os.path.basename(file_key)
                destino_local = os.path.join(path_silver_local, nome_arquivo)

                print(f"  📥 Baixando Parquet Silver: {nome_arquivo}...")
                s3_client.download_file(BUCKET_PRINICIPAL, file_key, destino_local)
        else:
            print(f"  ⚠️ Atenção: Nenhum arquivo encontrado na Silver para a entidade '{entidade}'.")
    except Exception as e:
        print(f"  ❌ Erro ao baixar dados de '{entidade}': {str(e)}")

print("\n✅ Processo de download em lote da Silver finalizado!")

--- [GOLD] Iniciando download em lote da camada Silver para as tabelas analíticas ---

📡 Buscando arquivos para 'alunos_2023' em: s3://fiap-datalake-fase2/silver/alunos_2023/anomesdia=20260627/
  📥 Baixando Parquet Silver: _SUCCESS...
  📥 Baixando Parquet Silver: part-00000-853f922e-bdad-450d-a426-b83836a60244-c000.snappy.parquet...
  📥 Baixando Parquet Silver: part-00001-853f922e-bdad-450d-a426-b83836a60244-c000.snappy.parquet...

📡 Buscando arquivos para 'alunos_2024' em: s3://fiap-datalake-fase2/silver/alunos_2024/anomesdia=20260627/
  📥 Baixando Parquet Silver: _SUCCESS...
  📥 Baixando Parquet Silver: part-00000-a91756d9-df58-48c0-addf-f8ccefa17cb5-c000.snappy.parquet...
  📥 Baixando Parquet Silver: part-00001-a91756d9-df58-48c0-addf-f8ccefa17cb5-c000.snappy.parquet...

📡 Buscando arquivos para 'alunos_2025' em: s3://fiap-datalake-fase2/silver/alunos_2025/anomesdia=20260627/
  📥 Baixando Parquet Silver: _SUCCESS...
  📥 Baixando Parquet Silver: part-00000-7e8179e7-6759-4aa2-886d-20d

In [18]:
# ==============================================================================
# CAMADA GOLD - CÉLULA 2: PROCESSAMENTO ANALÍTICO DEFINITIVO (IN_PRESENCA_LP = 1)
# ==============================================================================
import os
import shutil
from pyspark.sql import functions as F

print("--- [GOLD] Iniciando processamento definitivo na Camada Gold ---")

PATH_SILVER_BASE = "./output/silver"
PATH_GOLD_BASE = f"./output/{PASTA_GOLD}"

# 1. LEITURA DOS DADOS (Com colunas já em Maiúsculo vindo da Silver)
df_2023 = spark.read.parquet(f"{PATH_SILVER_BASE}/alunos_2023/anomesdia={ANOMESDIA}/")
df_2024 = spark.read.parquet(f"{PATH_SILVER_BASE}/alunos_2024/anomesdia={ANOMESDIA}/")
df_2025 = spark.read.parquet(f"{PATH_SILVER_BASE}/alunos_2025/anomesdia={ANOMESDIA}/")
df_meta_muni = spark.read.parquet(f"{PATH_SILVER_BASE}/meta_municipio/anomesdia={ANOMESDIA}/")

# Union dos três anos de alunos
df_alunos_all = df_2023.union(df_2024).union(df_2025)

# 🎯 FILTRO CORRETO: Filtrando apenas alunos presentes usando o número 1
df_base_calculo = df_alunos_all.filter(F.col("IN_PRESENCA_LP") == 1)

print(f"📊 Total de alunos unificados: {df_alunos_all.count()}")
print(f"🎯 Total de alunos PRESENTES para o cálculo: {df_base_calculo.count()}")

# ------------------------------------------------------------------------------
# VISÃO 1: Indicador de Alfabetização por Município
# ------------------------------------------------------------------------------
print("\n📊 Criando visão: indicador_alfabetizacao_municipio...")
df_indicador_municipio = (
    df_base_calculo
    .groupBy("NU_ANO_AVALIACAO", "SG_UF", "CO_MUNICIPIO", "NO_MUNICIPIO")
    .agg(
        F.count("ID_ALUNO").alias("TOTAL_AVALIADOS"),
        # Contamos como alfabetizado se o valor for 1 (ou '1')
        F.sum(F.when((F.col("IN_ALFABETIZADO") == 1) | (F.col("IN_ALFABETIZADO") == "1"), 1).otherwise(0)).alias("TOTAL_ALFABETIZADOS")
    )
    .withColumn("TAXA_ALFABETIZACAO_REAL", F.round((F.col("TOTAL_ALFABETIZADOS") / F.col("TOTAL_AVALIADOS")) * 100, 2))
)

# ------------------------------------------------------------------------------
# VISÃO 2: Comparação entre Metas e Resultados
# ------------------------------------------------------------------------------
print("🎯 Criando visão: comparativo_metas_resultados...")
df_metas_ajustadas = df_meta_muni.select(
    F.col("ID_MUNICIPIO").cast("string").alias("CO_MUNICIPIO"),
    F.col("META_ALFABETIZACAO_2024").alias("META_2024"),
    F.col("META_ALFABETIZACAO_2025").alias("META_2025")
).distinct()

df_comparativo_metas = (
    df_indicador_municipio.join(df_metas_ajustadas, on="CO_MUNICIPIO", how="left")
    .withColumn(
        "META_ANO",
        F.when(F.col("NU_ANO_AVALIACAO") == 2024, F.col("META_2024"))
         .when(F.col("NU_ANO_AVALIACAO") == 2025, F.col("META_2025"))
         .otherwise(None)
    )
    .withColumn("DESVIO_META", F.round(F.col("TAXA_ALFABETIZACAO_REAL") - F.col("META_ANO"), 2))
    .withColumn(
        "STATUS_META",
        F.when(F.col("META_ANO").isNull(), "Sem Meta Definida")
         .when(F.col("DESVIO_META") >= 0, "Atingiu a Meta")
         .otherwise("Abaixo da Meta")
    )
    .select(
        "NU_ANO_AVALIACAO", "SG_UF", "CO_MUNICIPIO", "NO_MUNICIPIO",
        "TOTAL_AVALIADOS", "TAXA_ALFABETIZACAO_REAL", "META_ANO", "DESVIO_META", "STATUS_META"
    )
)

# ------------------------------------------------------------------------------
# VISÃO 3: Evolução Temporal do Indicador (Nível Estadual)
# ------------------------------------------------------------------------------
print("📈 Criando visão: evolucao_temporal_estado...")
df_evolucao_temporal = (
    df_base_calculo
    .groupBy("NU_ANO_AVALIACAO", "SG_UF")
    .agg(
        F.count("ID_ALUNO").alias("TOTAL_AVALIADOS_ESTADO"),
        F.sum(F.when((F.col("IN_ALFABETIZADO") == 1) | (F.col("IN_ALFABETIZADO") == "1"), 1).otherwise(0)).alias("TOTAL_ALFABETIZADOS_ESTADO")
    )
    .withColumn("TAXA_ALFABETIZACAO_ESTADO", F.round((F.col("TOTAL_ALFABETIZADOS_ESTADO") / F.col("TOTAL_AVALIADOS_ESTADO")) * 100, 2))
    .orderBy("SG_UF", "NU_ANO_AVALIACAO")
)

# ------------------------------------------------------------------------------
# 2. ESCRITA DAS TABELAS OURO NO FORMATO PARQUET LOCAL
# ------------------------------------------------------------------------------
VISOES_GOLD = {
    "indicador_alfabetizacao_municipio": df_indicador_municipio,
    "comparativo_metas_resultados": df_comparativo_metas,
    "evolucao_temporal_estado": df_evolucao_temporal
}

for nome_visao, df_gold in VISOES_GOLD.items():
    path_gold_local = f"{PATH_GOLD_BASE}/{nome_visao}/anomesdia={ANOMESDIA}/"

    df_gold = df_gold.withColumn("_gold_processed_at", F.current_timestamp()) \
                     .withColumn("_analytics_version", F.lit("v2.1_gold_numeric_fix"))

    if os.path.exists(path_gold_local):
        shutil.rmtree(path_gold_local)

    print(f"  💾 Gravando parquet local da visão '{nome_visao}'...")
    df_gold.write.mode("overwrite").parquet(path_gold_local)

print("\n✅ Camada Gold gerada localmente com os filtros matemáticos corretos!")

--- [GOLD] Iniciando processamento definitivo na Camada Gold ---
📊 Total de alunos unificados: 6090791
🎯 Total de alunos PRESENTES para o cálculo: 5325767

📊 Criando visão: indicador_alfabetizacao_municipio...
🎯 Criando visão: comparativo_metas_resultados...
📈 Criando visão: evolucao_temporal_estado...
  💾 Gravando parquet local da visão 'indicador_alfabetizacao_municipio'...
  💾 Gravando parquet local da visão 'comparativo_metas_resultados'...
  💾 Gravando parquet local da visão 'evolucao_temporal_estado'...

✅ Camada Gold gerada localmente com os filtros matemáticos corretos!


In [19]:
# ==============================================================================
# CAMADA GOLD - CÉLULA 3: UPLOAD DAS VISÕES ANALÍTICAS PARA O S3 (GOLD FINAL)
# ==============================================================================
print("--- [GOLD] Iniciando upload das tabelas analíticas finais para o S3 ---")

ENTIDADES_GOLD_FINAIS = [
    "indicador_alfabetizacao_municipio",
    "comparativo_metas_resultados",
    "evolucao_temporal_estado"
]

for entidade in ENTIDADES_GOLD_FINAIS:
    path_gold_local = f"./output/{PASTA_GOLD}/{entidade}/anomesdia={ANOMESDIA}/"
    key_s3_gold_output = f"{PASTA_GOLD}/{entidade}/anomesdia={ANOMESDIA}/"

    if os.path.exists(path_gold_local):
        print(f"\n🚀 Enviando tabela de negócio '{entidade}' para: s3://{BUCKET_PRINICIPAL}/{key_s3_gold_output}")

        # Reaproveitando a função robusta de upload que você criou na estrutura original
        fazer_upload_pasta_s3(
            pasta_local=path_gold_local,
            bucket=BUCKET_PRINICIPAL,
            prefixo_s3=key_s3_gold_output
        )
    else:
        print(f"  ⚠️ Aviso: Caminho local '{path_gold_local}' não encontrado para upload.")

print("\n🏆 Parabéns! O pipeline da Camada Gold foi executado e armazenado com sucesso no S3!")

--- [GOLD] Iniciando upload das tabelas analíticas finais para o S3 ---

🚀 Enviando tabela de negócio 'indicador_alfabetizacao_municipio' para: s3://fiap-datalake-fase2/gold/indicador_alfabetizacao_municipio/anomesdia=20260627/

🚀 Enviando tabela de negócio 'comparativo_metas_resultados' para: s3://fiap-datalake-fase2/gold/comparativo_metas_resultados/anomesdia=20260627/

🚀 Enviando tabela de negócio 'evolucao_temporal_estado' para: s3://fiap-datalake-fase2/gold/evolucao_temporal_estado/anomesdia=20260627/

🏆 Parabéns! O pipeline da Camada Gold foi executado e armazenado com sucesso no S3!


In [20]:
# ==============================================================================
# CAMADA GOLD - CÉLULA 4: VISUALIZAÇÃO DOS RESULTADOS
# ==============================================================================
print("--- [GOLD] Visualizando uma amostra das visões analíticas geradas ---")

PATH_GOLD_BASE = f"./output/{PASTA_GOLD}"

VISOES_PARA_VALIDAR = [
    "indicador_alfabetizacao_municipio",
    "comparativo_metas_resultados",
    "evolucao_temporal_estado"
]

for nome_visao in VISOES_PARA_VALIDAR:
    path_local = f"{PATH_GOLD_BASE}/{nome_visao}/anomesdia={ANOMESDIA}/"

    print("\n" + "="*80)
    print(f"📊 TABELA OURO: {nome_visao}")
    print("="*80)

    try:
        # Lendo os arquivos parquet gerados localmente
        df_validacao = spark.read.parquet(path_local)

        # Exibindo o schema (estrutura) resumido
        print(f"📌 Total de registros gerados: {df_validacao.count()}")
        print("📋 Estrutura das colunas:")
        df_validacao.printSchema()

        # Exibindo as 5 primeiras linhas formatadas como tabela
        print("👀 Amostra das primeiras 5 linhas:")
        df_validacao.show(5, truncate=False)

    except Exception as e:
        print(f"❌ Erro ao ler a visão '{nome_visao}': {str(e)}")

print("\n✅ Validação das tabelas da Gold concluída!")

--- [GOLD] Visualizando uma amostra das visões analíticas geradas ---

📊 TABELA OURO: indicador_alfabetizacao_municipio
📌 Total de registros gerados: 15949
📋 Estrutura das colunas:
root
 |-- NU_ANO_AVALIACAO: integer (nullable = true)
 |-- SG_UF: string (nullable = true)
 |-- CO_MUNICIPIO: integer (nullable = true)
 |-- NO_MUNICIPIO: string (nullable = true)
 |-- TOTAL_AVALIADOS: long (nullable = true)
 |-- TOTAL_ALFABETIZADOS: long (nullable = true)
 |-- TAXA_ALFABETIZACAO_REAL: double (nullable = true)
 |-- _gold_processed_at: timestamp (nullable = true)
 |-- _analytics_version: string (nullable = true)

👀 Amostra das primeiras 5 linhas:
+----------------+-----+------------+-----------------------+---------------+-------------------+-----------------------+--------------------------+---------------------+
|NU_ANO_AVALIACAO|SG_UF|CO_MUNICIPIO|NO_MUNICIPIO           |TOTAL_AVALIADOS|TOTAL_ALFABETIZADOS|TAXA_ALFABETIZACAO_REAL|_gold_processed_at        |_analytics_version   |
+---------

In [ ]:
# ==============================================================================
# DATA QUALITY CHECK
# ==============================================================================